# Assignment 2:

Brain MRI triage for a general hospital with no neuroradiologist on site.
Each scan is routed to **pass** (normal report), **review** (remote
neuroradiologist) or **fail** (urgent referral).

This notebook downloads the dataset, describes it, finetunes `yolo11n-cls`,
compares it with two baselines, and picks the two thresholds that cost least.
Everything `app.py` needs is saved to `weights/` and `outputs/`.

In [5]:
!git clone https://github.com/nour-farhat/daiist-assignment-2.git
%cd daiist-assignment-2
!pip install -q "ultralytics>=8.4,<8.5" roboflow

Cloning into 'daiist-assignment-2'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 32 (delta 7), reused 3 (delta 3), pack-reused 14 (from 2)
Receiving objects: 100% (32/32), 83.51 KiB | 706.00 KiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/daiist-assignment-2/daiist-assignment-2/daiist-assignment-2


In [6]:
import hashlib
import json
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from roboflow import Roboflow
from ultralytics import YOLO

from api_key import load_api_key
import costs
from costs import NEGATIVE, GRID

# In Colab the key comes from Secrets; locally it comes from .env
try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
except ImportError:
    pass

ROOT = Path.cwd()
DATA_DIR = ROOT / "datasets" / "brain-tumour"
WEIGHTS_DIR = ROOT / "weights"
OUT_DIR = ROOT / "outputs"  # everything app.py reads
WEIGHTS_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# Roboflow Universe dataset, used as is (not forked)
WORKSPACE = "kaplan-business-school-gqxef"
PROJECT = "brain-tumour-detection-svrlh"
VERSION = 1

# Training settings
MODEL = "yolo11n-cls.pt"
EPOCHS = 20
IMGSZ = 224
SEED = 0

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [7]:
#downloading the dataset
if not DATA_DIR.exists():
    rf = Roboflow(api_key=load_api_key())
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("folder", location=str(DATA_DIR))

#ultralytics expects the validation folder to be called "val"
if (DATA_DIR / "valid").exists():
    (DATA_DIR / "valid").rename(DATA_DIR / "val")

SPLITS = ["train", "val", "test"]
CLASSES = sorted(p.name for p in (DATA_DIR / "train").iterdir() if p.is_dir())
print(CLASSES)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/daiist-assignment-2/daiist-assignment-2/daiist-assignment-2/datasets/brain-tumour in folder:: 100%|██████████| 5522/5522 [00:02<00:00, 2475.51it/s]


['glioma', 'meningioma', 'notumor', 'pituitary']


In [8]:
#Checking the dataset before training
def images(split, cls=None):
    """All image files in a split, or in one class of a split."""
    folder = DATA_DIR / split / cls if cls else DATA_DIR / split
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

# Images per class and split
counts = pd.DataFrame({s: {c: len(images(s, c)) for c in CLASSES} for s in SPLITS})
counts["total"] = counts.sum(axis=1)
counts.loc["total"] = counts.sum()
counts["share"] = (counts["total"] / counts.loc["total", "total"]).round(3)
counts.to_csv(OUT_DIR / "dataset_counts.csv")
counts

,train,val,test,total,share
glioma,894,281,146,1321,0.240
meningioma,928,281,124,1333,0.242
notumor,1003,248,155,1406,0.255
pituitary,1033,289,123,1445,0.262
total,3858,1099,548,5505,1.000


In [9]:
# Exact duplicate images across splits would leak the test set into training
hashes = {s: {hashlib.md5(p.read_bytes()).hexdigest() for p in images(s)} for s in SPLITS}
print("test images also in train:", len(hashes["test"] & hashes["train"]))
print("val images also in train: ", len(hashes["val"] & hashes["train"]))
print("duplicates inside train:  ", len(images("train")) - len(hashes["train"]))

test images also in train: 0
val images also in train:  0
duplicates inside train:   0


In [10]:
#finetuning yolo11n-cls
model = YOLO(MODEL)  # imageNet-pretrained classifier
model.train(data=str(DATA_DIR), epochs=EPOCHS, imgsz=IMGSZ, seed=SEED, patience=10,
            project=str(ROOT / "runs"), name="cls", exist_ok=True)

#commit this file: it is what app.py and check.py load
shutil.copy(model.trainer.best, WEIGHTS_DIR / "best.pt")
model = YOLO(WEIGHTS_DIR / "best.pt")

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/daiist-assignment-2/daiist-assignment-2/daiist-assignment-2/datasets/brain-tumour, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, n

In [11]:
#score validation and test images
#score = 1 − P(notumor), the probability that the scan shows any tumour. Thresholds are chosen on validation; test is used once, to report.
def score_split(split):
    """One row per image: true class, probability of each class, predicted class, tumour score."""
    rows = []
    for c in CLASSES:
        for r in model.predict(images(split, c), imgsz=IMGSZ, verbose=False, stream=True):
            probs = r.probs.data.tolist()
            rows.append({"file": Path(r.path).name, "true": c,
                         **{model.names[i]: p for i, p in enumerate(probs)}})
    df = pd.DataFrame(rows)
    df["pred"] = df[CLASSES].idxmax(axis=1)
    df["score"] = 1 - df[NEGATIVE]
    return df

val_df, test_df = score_split("val"), score_split("test")
val_df.to_csv(OUT_DIR / "val_scores.csv", index=False)
test_df.to_csv(OUT_DIR / "test_scores.csv", index=False)

In [12]:
#Per-class results on test
cm = pd.crosstab(test_df["true"], test_df["pred"]).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
per_class = pd.DataFrame({
    "images": cm.sum(axis=1),
    "precision": (np.diag(cm) / cm.sum(axis=0)).round(3),
    "recall": (np.diag(cm) / cm.sum(axis=1)).round(3),
})
accuracy = np.trace(cm) / cm.values.sum()

cm.to_csv(OUT_DIR / "confusion_matrix.csv")
per_class.to_csv(OUT_DIR / "per_class.csv")
print(f"4-class accuracy: {accuracy:.3f}")
print("rows = true class, columns = predicted class")
cm

4-class accuracy: 0.976
rows = true class, columns = predicted class


pred,glioma,meningioma,notumor,pituitary
true,,,,
glioma,140,6,0,0
meningioma,2,121,1,0
notumor,0,2,152,1
pituitary,0,1,0,122


In [13]:
per_class

,images,precision,recall
glioma,146,0.986,0.959
meningioma,124,0.931,0.976
notumor,155,0.993,0.981
pituitary,123,0.992,0.992


In [14]:
#Baselines on the same test split
#COCO zero-shot: yolo11n.pt has no tumour classes, so it cannot name a tumour type. The best we can do is use its highest box confidence as a "something is there" score.
#Trivial: always predict the majority class of the training set. For the routing decision, the trivial rule is "pass everything" (do nothing).
def auc(df):
    """Chance that a random tumour scan scores above a random healthy scan."""
    ranks = df["score"].rank().to_numpy()
    tumour = (df["true"] != NEGATIVE).to_numpy()
    n1, n0 = tumour.sum(), (~tumour).sum()
    return (ranks[tumour].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# COCO zero-shot: highest detection confidence on the image, any class
coco = YOLO("yolo11n.pt")
coco_df = test_df[["file", "true"]].copy()
coco_df["score"] = [float(r.boxes.conf.max()) if len(r.boxes) else 0.0
                    for c in CLASSES
                    for r in coco.predict(images("test", c), verbose=False, stream=True)]

majority = counts.loc[CLASSES, "train"].idxmax()

baselines = pd.DataFrame([
    {"model": "finetuned yolo11n-cls", "accuracy": accuracy, "tumour AUC": auc(test_df),
     "best cost / 1,000 (no review)": costs.best_single(test_df)[1]},
    {"model": "COCO yolo11n zero-shot", "accuracy": np.nan, "tumour AUC": auc(coco_df),
     "best cost / 1,000 (no review)": costs.best_single(coco_df)[1]},
    {"model": f"trivial: always '{majority}'", "accuracy": (test_df["true"] == majority).mean(),
     "tumour AUC": 0.5, "best cost / 1,000 (no review)": costs.cost_per_1000(test_df, 1.01, 1.01)},
]).round(3)
baselines.to_csv(OUT_DIR / "baselines.csv", index=False)
baselines

,model,accuracy,tumour AUC,"best cost / 1,000 (no review)"
0,finetuned yolo11n-cls,0.976,1.000,3816.794
1,COCO yolo11n zero-shot,NaN,0.538,570000.000
2,trivial: always 'pituitary',0.224,0.500,1500000.000


## 7. Cost the decision and thresholds

Costs and base rate are in costs.py. The dataset is mostly tumours, but the
hospital is not, so every cost is computed at the hospital's BASE_RATE.

In [ ]:
# Model alone: one threshold, swept on validation, also shown on test
sweep = pd.DataFrame({
    "threshold": GRID,
    "val_cost": [costs.cost_per_1000(val_df, t, t) for t in GRID],
    "test_cost": [costs.cost_per_1000(test_df, t, t) for t in GRID],
})
sweep.to_csv(OUT_DIR / "cost_sweep.csv", index=False)

# Model + review band: cheapest (t_low, t_high) on validation
(t_low, t_high), _ = costs.best_band(val_df)
t_single, _ = costs.best_single(val_df)
print(f"model alone: threshold {t_single}")
print(f"with review: pass < {t_low} <= review < {t_high} <= fail")

plt.plot(sweep["threshold"], sweep["val_cost"], label="validation")
plt.plot(sweep["threshold"], sweep["test_cost"], label="test")
plt.axvline(t_single, color="grey", linestyle="--", label=f"chosen single threshold {t_single}")
plt.axvspan(t_low, t_high, color="orange", alpha=0.2, label="chosen review band")
plt.xlabel("threshold on tumour score")
plt.ylabel("expected cost per 1,000 scans (EUR)")
plt.yscale("log")
plt.legend()
plt.savefig(OUT_DIR / "cost_curve.png", dpi=120, bbox_inches="tight")

In [16]:
#who still has to check: the four options, priced on test:
share = costs.review_share(test_df, t_low, t_high)
reviews_per_day = share * costs.SCANS_PER_DAY
readers = reviews_per_day * costs.MINUTES_PER_READ / 60 / costs.READER_HOURS_PER_DAY

options = pd.DataFrame([
    {"option": "do nothing (pass everything)", "cost / 1,000": costs.cost_per_1000(test_df, 1.01, 1.01), "sent to a person": 0.0},
    {"option": "person checks everything", "cost / 1,000": 1000 * costs.COST_REVIEW, "sent to a person": 1.0},
    {"option": f"model alone (t = {t_single})", "cost / 1,000": costs.cost_per_1000(test_df, t_single, t_single), "sent to a person": 0.0},
    {"option": f"model + review band ({t_low} to {t_high})", "cost / 1,000": costs.cost_per_1000(test_df, t_low, t_high), "sent to a person": share},
]).round(3)
options.to_csv(OUT_DIR / "options.csv", index=False)
print(f"{reviews_per_day:.1f} reviews a day -> {readers:.2f} neuroradiologists at {costs.READER_HOURS_PER_DAY} h/day")
options

0.6 reviews a day -> 0.03 neuroradiologists at 6 h/day


,option,"cost / 1,000",sent to a person
0,do nothing (pass everything),1500000.000,0.000
1,person checks everything,50000.000,1.000
2,model alone (t = 0.63),7494.213,0.000
3,model + review band (0.1 to 0.77),5349.052,0.031


In [17]:
#sensitivity: when does the answer change?
#re-pick the band on validation for other base rates and other costs of a missed tumour.

rows = []
for base_rate in [0.01, 0.02, 0.05, 0.10, 0.20]:
    for cost_fn in [5_000, 15_000, 30_000, 60_000]:
        (lo, hi), _ = costs.best_band(val_df, base_rate, cost_fn)
        rows.append({"base rate": base_rate, "cost of a miss": cost_fn, "t_low": lo, "t_high": hi,
                     "test cost / 1,000": costs.cost_per_1000(test_df, lo, hi, base_rate, cost_fn),
                     "share reviewed": costs.review_share(test_df, lo, hi, base_rate)})
sensitivity = pd.DataFrame(rows).round(3)
sensitivity.to_csv(OUT_DIR / "sensitivity.csv", index=False)
sensitivity

,base rate,cost of a miss,t_low,t_high,"test cost / 1,000",share reviewed
0,0.01,5000,0.77,1.00,625.954,0.010
1,0.01,15000,0.63,1.00,1199.762,0.016
2,0.01,30000,0.63,1.00,1581.441,0.016
3,0.01,60000,0.10,1.00,3622.219,0.042
4,0.02,5000,0.63,1.00,1568.037,0.026
5,0.02,15000,0.63,1.00,2076.943,0.026
6,0.02,30000,0.10,1.00,4104.818,0.052
7,0.02,60000,0.10,1.00,5631.536,0.052
8,0.05,5000,0.63,0.77,942.584,0.006
9,0.05,15000,0.10,0.77,3440.655,0.031


In [18]:
#save the operating point for the app
metrics = {
    "dataset": f"{WORKSPACE}/{PROJECT}/{VERSION}",
    "settings": {"model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "seed": SEED},
    "accuracy": accuracy,
    "tumour_auc": auc(test_df),
    "t_single": float(t_single),
    "t_low": float(t_low),
    "t_high": float(t_high),
    "test_cost_per_1000": costs.cost_per_1000(test_df, t_low, t_high),
    "review_share": share,
    "reviews_per_day": reviews_per_day,
    "readers_needed": readers,
}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
metrics

{'dataset': 'kaplan-business-school-gqxef/brain-tumour-detection-svrlh/1',
 'settings': {'model': 'yolo11n-cls.pt',
  'epochs': 20,
  'imgsz': 224,
  'seed': 0},
 'accuracy': np.float64(0.9762773722627737),
 'tumour_auc': np.float64(0.9998686694574407),
 't_single': 0.63,
 't_low': 0.1,
 't_high': 0.77,
 'test_cost_per_1000': np.float64(5349.051957645901),
 'review_share': np.float64(0.03064516129032258),
 'reviews_per_day': np.float64(0.6129032258064516),
 'readers_needed': np.float64(0.025537634408602152)}